# Object Detection


This lab is similar to the previous lab, except now instead of printing out the bounding box coordinates, you can visualize these bounding boxes on top of the image!

## Setup


In [ ]:
# For running inference on the TF-Hub module.
import tensorflow as tf

import tensorflow_hub as hub

# For downloading the image.
import matplotlib.pyplot as plt
import tempfile
from six.moves.urllib.request import urlopen
from six import BytesIO

# For drawing onto the image.
import numpy as np
from PIL import Image
from PIL import ImageColor
from PIL import ImageDraw
from PIL import ImageFont
from PIL import ImageOps

# For measuring the inference time.
import time

# Check available GPU devices.
print("The following GPU devices are available: %s" % tf.test.gpu_device_name())

### Select and load the model
As in the previous lab, you can choose an object detection module. Here are two that we've selected for you:
* [ssd + mobilenet V2](https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2) small and fast.
* [FasterRCNN + InceptionResNet V2](https://tfhub.dev/google/faster_rcnn/openimages_v4/inception_resnet_v2/1): high accuracy

In [ ]:
# you can switch the commented lines here to pick the other model

# ssd mobilenet version 2
module_handle = "https://tfhub.dev/google/openimages_v4/ssd/mobilenet_v2/1"

# You can choose inception resnet version 2 instead
#module_handle = "https://tfhub.dev/google/faster_rcnn/openimages_v4/inception_resnet_v2/1"

#### Load the model

Next, you'll load the model specified by the `module_handle`.
- This will take a few minutes to load the model.

In [ ]:
model = hub.load(module_handle)

#### Choose the default signature

As before, you can check the available signatures using `.signature.keys()`

In [ ]:
# take a look at the available signatures for this particular model
model.signatures.keys()

Please choose the 'default' signature for your object detector.

In [ ]:
detector = model.signatures['default']

### download_and_resize_image

As you saw in the previous lab, this function downloads an image specified by a given "url", pre-processes it, and then saves it to disk.
- What new compared to the previous lab is that you an display the image if you set the parameter `display=True`.

In [ ]:
def display_image(image):
    """
    Displays an image inside the notebook.
    This is used by download_and_resize_image()
    """
    fig = plt.figure(figsize=(20, 15))
    plt.grid(False)
    plt.imshow(image)

def font_getsize(font, text):
    """
    Get text width and height for a given font.
    This is used by draw_bounding_box_on_image()
    """

    left, top, right, bottom = font.getbbox(text)

    return right - left, bottom - top


def download_and_resize_image(url, new_width=256, new_height=256, display=False):
    '''
    Fetches an image online, resizes it and saves it locally.

    Args:
        url (string) -- link to the image
        new_width (int) -- size in pixels used for resizing the width of the image
        new_height (int) -- size in pixels used for resizing the length of the image

    Returns:
        (string) -- path to the saved image
    '''


    # create a temporary file ending with ".jpg"
    _, filename = tempfile.mkstemp(suffix=".jpg")

    # opens the given URL
    response = urlopen(url)

    # reads the image fetched from the URL
    image_data = response.read()

    # puts the image data in memory buffer
    image_data = BytesIO(image_data)

    # opens the image
    pil_image = Image.open(image_data)

    # resizes the image. will crop if aspect ratio is different.
    pil_image = ImageOps.fit(pil_image, (new_width, new_height), Image.Resampling.LANCZOS)

    # converts to the RGB colorspace
    pil_image_rgb = pil_image.convert("RGB")

    # saves the image to the temporary file created earlier
    pil_image_rgb.save(filename, format="JPEG", quality=90)

    print("Image downloaded to %s." % filename)

    if display:
        display_image(pil_image)


    return filename

### Select and load an image
Load a public image from Open Images v4, save locally, and display.

In [ ]:
# By Heiko Gorski, Source: https://commons.wikimedia.org/wiki/File:Naxos_Taverna.jpg
image_url = "data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAkGBwgHBgkIBwgKCgkLDRYPDQwMDRsUFRAWIB0iIiAdHx8kKDQsJCYxJx8fLT0tMTU3Ojo6Iys/RD84QzQ5OjcBCgoKDQwNGg8PGjclHyU3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3Nzc3N//AABEIAJQA6gMBEQACEQEDEQH/xAAcAAEAAQUBAQAAAAAAAAAAAAAABQECAwQGBwj/xABJEAABAwMCAgYFCAUICwAAAAABAAIDBAURBhIhMQcTIkFRYRRxkaHBFTJCUnKBsdElYpKi4RYjJDNTVGOCFzQ1Q0RFc5Oy8PH/xAAbAQEAAwEBAQEAAAAAAAAAAAAAAQIDBAUGB//EAD0RAAIBAgMEBgcGBQUBAAAAAAABAgMRBBIhBRQxURMyQXGRoRUiYYGx0eEjQlJTwfAkNENi0hYzVJKiBv/aAAwDAQACEQMRAD8A9xQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQFCQOZQGKWqp4hmSeNo83gKLllCT4IwMudFJOIGVkDpXDcGCQZI8Uui3RVLXaNwclJmVQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAygKOcGjJIA80BpS3OkidtdKCfIZUXNY0Zy4IwPvlI3lvd6gozIusLUNabUtPE0uMe1vi54Cq6iRvDZ9Sb0IWu6QaWmzsa158G8fesniF2HfT2JKXWdjna7pIrXPd6O0Rg8u/H3KjrzfA7qex8NFevqQ9TrW81J/1uVvkDge5ZupN9p1wwOEhwgiCuVzr6mZsklfUgObxHXOxn2qFN8zR0KUX6sV5GhKWPGZpNx8XHKjR8S6SS4m3ZpXVV0YXTue5sTgHF5O32qbtmXRxS0R1NPeLtRH+jV8o/VLiQrqpJHJUwlKfGKZLUuvbxBgVEUM7R3kYPuWirNcThqbMpPhoTdH0jUb8Cso5Yj3lh3BaKsjknsqS6rJyj1XZavGyujYT3SdlXVSLOSeCrw+6TMM8M7d0MrHjxa4FXvc53GUeKL8hCoyEAygKoAgCAIAgCAIAgCAIChOEBE3q+UtqhDnl0kj3bWMjBcc+eOQVXNI3o4adWXDQ46v1PPVOy9pa0cmueGAe0rF1GevSwajwi2Rr76GtO6SlB7t0wP4Kuf2nTHDSb6hHS6ikPZFTTR+bQ9+P3R+KzcvadUMPb7nnY1o6N13ZJM7UMEQZ84ywuAx7VVJSfE3q1ZUFpT09jKnRdY5m+O60LyeI3NeFbozFY6V9Yv8AfvNKbSF/Zkwm3TfZmx+OFGS37+hffI87d6f1NOax3mkYH18cdPnljt5+8FVat2GtOo6nVkn4/rYiL/FJFTwufK15D8cG47v4KFZl62aKTbIcPKtYyUif0ed1zP2D+CgutTrJG80IaMLmqTNoxuB8vYpKOJYY2/VblCuUujkmhcHQzyxnuLHEKVJlJQT4ktR6ovlJjZXve0fRl7QV1No5p4OjL7pN0fSFXMAFZRxSebDhaKs+05J7Mh912Jqj6QLXKdtRFPA7vy3cPcrKqu05JbOqLgyco9RWesAMFwgJP0XO2n2FaKSZzzw1WHGJJNe14yxwcPEHKkxaa4lcoQVQBAEAQBAEAQHH9IepKW026SiFSWVs7BtYw9oMOQXDw5Ks1K2h0YXIqydR6HjJri/jiqeTzxG45XNu9Vn0PpfCRVrvwLDPKfm0VSftNDfxKndplHtyguEH5Fjp6kcqIDzfM0K26vtZlPb34afn9DG64VtOx5aIGNe0tcGy7iR4K27JdpjLblSX9NeL+hI6cuTpJPRZ6eJ0MmQ45xgY8+CpUoRjFyTNqG1amKqqlKCszsPTGDg1wOPArmbPZVLTQyR1pzxJ9qKViroFtznbU0W13HaeCmTuhQpZJ3RwepWYoXHh2Xt/HCrDibYr/bucwCtLHHFnQ6NOLofsH8CqM6afadi9Rcs0YSrFGixSUaLUK2CkhophCtiu0HxVithjCFGihaM5whGU2Ketq6U5p6qaL7DyrKTMpUoy4o6fSmpLtU3qko6mq62KRxB3NGcYPetITbdjgxWFpxpOSWp6UOS3PHKoAgCAoUBD3vUdJZnsjmp7hUSyN3NZSUUsxx5loIH3lAX/ACtO63enCgdDHt3FtXIInNHmOOPvQHjGuLzXV15lrWwVrIZOwwRxuIG3DSPb4gZ5q97IrZN6kBNHffp2S+u9dHL+SXJsiPkfcy4t+Sa0OBwRKwsI9qgaGMw3l3zbW8euRo+KWY0LPk6+SfOpIoh4vlBx7EsxdErY6KrNU6lElO6pcGhskkIc2PPE8Dz4LKp1lFno4JWo1aq4pIlJLjQUldLb7lBRSTQv2On9EbtJ4eA81WVBX0ZpS2pJRvNX95OS2+npTCZaKGNspw10Er2jlnkCO4Kk6Tgrs7MNj1iKmSLafeUqbcxrHdS+tiZnBcJt7R54cSVhdHpxdVPWRzupLe+mo6pj6h8u0A9oDHPPgqrib1HOVB5mca09y0ZxQZ0Ojj+k/wDKfwKykdtDW52byqm7RhJU3M7FqsUaKFCGiikpYKSGghWxVTcq0FJWxQoVaJbR5xqm3echH7pV4dZHHjP9iR7EOS6j54qgCAIAgCA43pbqn0WgrlJE4se4MYC04Iy4Dh92UBEaJhdWTzvuQbPJHTxB5kG7MjhuJ4/+8FfsM3xO1z4IQeSa4qnHVNaPDY390KyJIVtT4qQY55y5oGSoYLtLu3Xx8nPE2P2WALnlrWij1qPq7Mqvm0vgcpd6gVGoas57UlU5oGeJO7C14s81dU9NvJxS0Q+pI1v7pCrXX2bOnZb/AIqC7/gZZXdZbqhg5uY8e0H815x9go6kHqVvXUdSR9KHPuULiaSX2MzzppytWeZBnQ6ROLifV8Cspno4ZcTsHuVDpaMZcpKNFpcpKNDKlEWGVJSwBQrYrlTciwyhRoZU3IsUJU3KNEppN2NUWw/4uP3Srw6yOPGL7CR7KF1nzZVAEAQBAEBxPS6yKXSTYJWuImradg2+O8c/LAKlAiNO3u22v5SfVTO3vnG1sUTnktAx3BWZmbbdasdI97YGupm8sE9ZjzbhTYg4W9Tx3K61Nc7roxM7cGGM8BjHwUokw08Nsk4TVro3eD8N+KkgrUQ2eAg+nRnB7pGlRpwCTZG6ZnHpTZImlxkkkeB9YE4HuAXKpfbnuOnL0UrLVu/nb9CLozfXXWea3UdVLmpflwpyWHtY5kY962cknqzyVSlJaJneV1DX1MEAdT4IkY925wby58yq1pxcGkzs2fQqRxMZyjoZYKebqdr+qAe0cTOzhwA8V52U+tdSPYn4MibuwtonRkhxFPtJHIkZVe1G61hJLkeaMPJdDPGpsn9LO215Pl+axmerg1ds6t83isztcTEZx4hSZuJQTDxHtUlMpXrR4hCLFRL5qSlh1ikrYr1iFbFesUlbDrEKtFOsUlbElpZ+dT2v/r/Aq9PrI48Yv4eZ7cF2nywQBAEAQBAc50gvczSVwcwgO2Da7GdpyBlRKWVXN8NR6aqqfM8DnoblPxFexoPiHnPvWG9LkeutgVPzF4GuLFW79zrgw55/zZ/NRvS5FvQE3/U8vqVfYp3fOrGD1QfxTe1yLL/5+S41PL6mL5Ak476x5B7urAwo3rki62Cu2o/AuitMLJQJZJJsdziAPcqyxMpLTQ3o7Ew8Z+u3K3MzXTZVlsIc6PZwDozgtx4LmjPK7nsVsKqyyKTitOGljZttQKCmbCzc/Bzue7JJPepnUcnc6MLgKeHpKEW/f2mybvPg4a0epZ5/adHQRLKe8VETAwOBHeSEUzKVBS4aF9dcnzUzmOLS4sPaaOSnNwCoOObuOHj5AeC7JHylInNPO21efL81z1D3Nnq82T8sxWR6Moms6Yqxm4lonOVJRxLxOUKZS8T+akrlLxUeakrlK9f5oVylfSPNTcq4lPSPNCriOv8ANSVaJPSs+dTWzH94b8VeHWRx42P8PPuPfAu4+QCAIAgCAIDgemG/stFhhojCZZLlJ1TMOwGbcEk+5Z1eq0d2z9MRGR4k++1AJDKfcPI5+Cy3Zcz1PT8ou3Ref0MLtR1IPGld7Hfkm6rmP9RVPy14/Qt/lJUn/hiP8rvyTdVzK/6hqflrx+hli1DVRv6x1LC79WYEj2KVhlzKT29UmrOFu5/Q37dc4Kvrai4U20ud2WUvYb7FzVkoSse7syrUxFDpOHfr5l1e+gkfE6gp5YCGnrOsfu3HuPkuaTTR6lGE4ybm7mtkql2diYNFX10rI7ayR7h85sbST+C6cO4XeZXPC2ysQ1Doaigtb9hLUul73JG0m3zk45naM+9RKjKTdkWp7TpU6cIyldpK5sVmn726nY35G2dVGWl7MBz/ADd2uJUujLTQtS2hRjnbk3fn+h5+xjmnBHEEjC6pHhUkTNiyKnHeR+a5qh72zuuyclY88mOJ9SyPSk1zNd0ch4bH58NpVrGbceZiLJeOI5Dj9UoVbjzAbL/Zv/ZKkpePMuxJ9R+fslCunMr/ADn1H5+yVJGnMqBKeUcn7JQroV2zf2Uv7BUldOa8SvVzDiYpB62FCjtzGybOOrf+yVJR25kjpjrI9R2xzmuAFXGCdp7yrw6yOXFpPDzXsZ9EDku8+LKoAgCAIAgPO+mHSl01NTWqSzsjkkopnukje8NJa4DiCeHDCpNXR1YSoqdRNvg0zwesjZSTllTUOhcSeHVuzz9iiM3a1jevhaedydVJPnm/RGq4xH/mjR64ipzv8LM1hKb/AK0f/X+JTDAf9ps/7RTpP7WRusPzo+f+JdI6IDb6bG7zcCPgnSP8LG6x4qrHz+RvQzNigjYyOSQbchzWHBXFOlKcm3ofUYPG0sPQjTSbsuNuJLWiTrnTROtss0pjG1pacsznte5Z9E4rmdXpCFWaV3G2trNmwLRcAWl9FUMjc4AvczAGfFZdDJcTu3+jJ2i7vuOguGoodLULaS3tax+A6V548fiVvFy6kOJ5laFNp4jFStH98Dmz0h1kjtgrpYhn6MQACu6Nexy08fsnNlafe1p8TO67XGoG6W4VDmn/ABSAVyXm3Y+kp4XDRWZRVuZw762Jkr8xuJ3HJB58V6fRSa4nwqx1KnN3j2s37Vd4YZ93VS8u7H5rOWGk+DO/DbcoUW24vy+ZI/yia0HDZDucXbcjs8APgq7rLmavb+HcnJU5O5RmoYwC50JfuPFhfjAHerbs7WuZS25SfrKD7rlrtRxZL2RCJ2MYE2c+0Juz5lPTlNetlfdcxDULQS7a0kgc5P4Kd29pHp2F79H5mM6leJN7KcuO3GA9Tu1u0o9uRbuqfmWO1M8kydR2iMY3nCjdvaR6bV79H5lrdQzF/wDNwZLncusKtuy5lFtuSb+zXiS8k9VBE5057bRl3VRue2M+BP5Juq5j07P8teZpvr7i8FraVsrcBzXxElrh5FN3XMpLbdR/cR09iis1RbRV6huNwt87XOBbDRmRjADzJwVO7x5lHtirdPIiQ0dZ57lrIC3yult8E8c7ah8Za58I4h+DjgSMck6GzTRZ7VlOE1JJXR743ktzxiqAIAgCAIDj+kyqqaa221tLWT0gnuEcU0sBw7qyx5I5HvAUNX0NqE1DNKydl296PFdVSTUF7kpYWtqoRgskqIA52HAHjwHiVgorM4uVvee1PFVHRp1IU4yzXvo3ZruIV1Xc2uI+S6RwB/uyt0S/Ezm32r24eP8A1kWmurM4Npovvp1PRP8AEyN/50I+BfDVCeYwut9Fy45hHs9qzqpwjfMzqwNeniauWVGKSTbfcd9PoukprS6tqLhGyWOEv6uOJm0EDO3nlJUUo3bKUdqSnWjCNONm7HKtutypohNQyCGZnASRxAO2945ZwuOnN5rX0PpsVh1KlmSu17/Iz2jUF4rKwx3G5zvia0Exk4DvIrSbSWlzkwlGUqjUktOSsRVVDDdrlNNWucWB3ZG/AHiV1YaGWF32nz228V0uKdNdWGi/V+Jo1drtkufk6SRsg5B3FrveSuk8g06H+kNkpqh0gMERLWbsDIPIrmq+r6yPa2e3i4yo1ZNqKbSvpp2E/TWbTs1FTyysq2yuZl4a/vI7uK55YiSdmz2KWw6VSCqKCs0u1/Muls2nzjqIaxju49bkH7sKu9PmW9AU5W9VL3v5m1ZNOWCtfcqiubWxwQSxRxxwOB+eO/I8R3LtoyzwufN7Uw0cNinSjw0+BJ01n0TLljLddZS0A7nPa3n6sLWx592WT2vSLLWayOwVrx1r2Fj6sh3ZIGfncuKkXZA11Xpeiax0OmhIHEjtVjjjGD8VAKUd0tE1LXTR6YoGmmhEmHyOduy8N8PAk/cpBgj1JC54bT6aszT3Axk5UEmWukgqblZJ208UD5qds0jIWBsfznd3+VFxJeXLx1/fmbU8hne2njc8zBpkwDhoHq7yVJUlOjiWf0242mmE75ARNFFBDTvcQRxA63wzyChg68WjUTLbcYJbdc9srpSxjKWjO4OH0jnIz5cvJATPRvbrlRVNKK2lraZsVu6gtnhYAMPyGtc3OcZPPihN1ax6MOAUEFUAQBAEAQHH9KMU0mnIzSxGaeOshfHGMdo7sY4+RKrLNplRvQ6Ntqo7Kx4rresjudye+4UclHNDGxkseQXBw7+HkQuZzn0t0j36WGw8sC4Tk8t73XYct6NazJxr5o8/Wb+QWmaf4Dh6DCPq4n4l4goWvxHd5PWMjPvCjM/yzRYeH/LXi/mSdmNHDVwyVFRJPTCZjpnDidoOSAPUsJyvUV42PTw2Hy4WahVzuXbrZfE72+6pbVacq6K122vEcn9ZUTM2hrTzyAe8LarUlKHA83AYSFPEJyqJvkuZworKugibUW+TZKxwILTg48MrkpWzWPosapSw91G6vw4X8GY/5RXyvc6F8znMIy/rJjtK6J5betJnj4aVZVb4bDxUl23+bM2maaO43OGmnmhij7Uj3TZ2kjjg48V2q1lY+bqOTm3Ljdk7qT5IqbhW26hp5oaulYZWmR/Z3DiYxnuORgDh2vJSUOKib+l5SOT4nH3fwWGI6nvR62xf5q3OMvgTNuP6OpuP+7H4LzqnXZ99s1/wVN/2ozlrs/Nd68KiNnON+JM6WbTTw3WCprYaN/XUzml7SS7aCe7vXp4bqeJ+e7e/npa30Rf6Rpai6wy6m60uAGIaVx5E+R7yug8Y0p7/AKSbRNoo6q51DA98mWQhpJcRnnj6qEm3o/TNl1m+ZlrttYYqV4MstXU7Rl3gBnJwOXBGCdrOj6vttVJSWrS1LWU9Q3Y6odXYYW5B7TTxHEdwKi4ITXVsGiJbe2WxWef0tjiHNY4hjm4yMnn85LixyFVqae4mKOS30VO2B2G9RGGkDiMZ8OJKXB1WkbSaivrbnMGCkpoMPc48nf8AzJUg1ejOojHSnQbHNw9z2AZ4/wBWTj3KGD6SUArhAEAQBAEAQBAQmsLLLf7FPQU9T6NM4tdHKW5Ac05GfLgqyWZWNqFboZ57XPK6jo11cJHP/RNW48C6QcXe0LB0He9z2Ke14RWVx0Im56LbYLfLcdVWRjYGyNb1lC/ssB4DIB4ce/HetYJqNpHn16tKc06SSXtRydzqdKh8ZtdLIQd3WNnkeMDhtwQR+som5fdL4XdpNqul7LafAtZdbYI2Qx0EbQ08Nkzsk/fnK5nTm3dntU8bhaUFTg9Ds6is1hVUU1IbFXinnGJGtp8FwwBz255AK8lVkrHNTns+nNVEtU78WR0dhuT2CGTS1y2H5xG4O+4kY9yyjQlmTaO+ptSjKi4RnYwXi2w2KF1TWWO4UTJjsjkqJuzuxywG8Tw8VpOm2kkn4nPhcbTpTc5VI+6L+Zz9jqnwzSFm0udHgbhyz3/cu1cD5eTu2zsG+isoqu+RF8VTUxmERbOrDHu4HYMcW+feB5qSDiqMxSaiiifIGMke2DcBnbu4cvLKzqxco2R27PxEcNX6SXJ+aPZbd0SVUMDoJr6BGxrWwOig5jH0gT4+CxeGi3ds7IbbrQpxpxitFbt+Zvw9ElCI4DUXe4SStLTNgtDJPEAYyPapWHgmUltrFSXYvcdJpzRdr05X1VXazUs9Ja1skMku9mW8nDPHPE962jFRVkedWrTrSzzd2Q176ItK3ivmrZIaqmmmfvk9Gm2tc48zgggZ8lJkRbugvS5fuFdeQPq9fHj/AMMoDudK6btulrYLfaYXMiyXvc92573Hvce8oCZQHN660hSaxtIoqmV0EsT+sgnY0Exu9XeD3hAeYu6Crg2Rzor5SkHudTuHxQG7RdD11ZK1lTeoG07iBN1LHbnN8s9/rypuD1OisdsoeqNLQU8b4hhkgjG4cMZyoBIjmUBVAEAQBAEAQBAEAQGCspIK2lmpauFk8EzSySOQZa4HuIQHm9w6ENMVMzpKeWtpWniI2SBwb6shASOmeibTWn62OubFJWVURzG+qduDHDiHBvLI8UB3gB70BXCAiNU6do9T2We1XEHqpMOa9p7UbxycPUgPEaroi1ZQVJbQtpK2JhPVTCcMJb4Oa7+KAm9O9Fl8qa6F2on0tNRMIL4oZese8fVBxho9Sm4PRf8AR/pLdu/k/bw4HIcIsHPrUA6Vo2tDQMADAQFUAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQFMcUAQFUAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQBAEAQH//2Q=="  #@param
downloaded_image_path = download_and_resize_image(image_url, 1280, 856, True)

### Draw bounding boxes

To build on what you saw in the previous lab, you can now visualize the predicted bounding boxes, overlaid on top of the image.  
- You can use `draw_boxes` to do this.  It will use `draw_bounding_box_on_image` to draw the bounding boxes.

In [ ]:
def draw_bounding_box_on_image(image,
                               ymin,
                               xmin,
                               ymax,
                               xmax,
                               color,
                               font,
                               thickness=4,
                               display_str_list=()):

    """
    Adds a bounding box to an image.

    Args:
        image -- the image object
        ymin -- bounding box coordinate
        xmin -- bounding box coordinate
        ymax -- bounding box coordinate
        xmax -- bounding box coordinate
        color -- color for the bounding box edges
        font -- font for class label
        thickness -- edge thickness of the bounding box
        display_str_list -- class labels for each object detected


    Returns:
        No return.  The function modifies the `image` argument
                    that gets passed into this function

    """
    draw = ImageDraw.Draw(image)
    im_width, im_height = image.size

    # scale the bounding box coordinates to the height and width of the image
    (left, right, top, bottom) = (xmin * im_width, xmax * im_width,
                                ymin * im_height, ymax * im_height)

    # define the four edges of the detection box
    draw.line([(left, top), (left, bottom), (right, bottom), (right, top),
             (left, top)],
            width=thickness,
            fill=color)

    # If the total height of the display strings added to the top of the bounding
    # box exceeds the top of the image, stack the strings below the bounding box
    # instead of above.
    display_str_heights = [font_getsize(font,ds)[1] for ds in display_str_list]
    # Each display_str has a top and bottom margin of 0.05x.
    total_display_str_height = (1 + 2 * 0.05) * sum(display_str_heights)

    if top > total_display_str_height:
        text_bottom = top
    else:
        text_bottom = top + total_display_str_height

    # Reverse list and print from bottom to top.
    for display_str in display_str_list[::-1]:
        text_width, text_height = font_getsize(font, display_str)
        margin = np.ceil(0.05 * text_height)
        draw.rectangle([(left, text_bottom - text_height - 2 * margin),
                        (left + text_width, text_bottom)],
                       fill=color)
        draw.text((left + margin, text_bottom - text_height - margin),
                  display_str,
                  fill="black",
                  font=font)
        text_bottom -= text_height - 2 * margin


def draw_boxes(image, boxes, class_names, scores, max_boxes=10, min_score=0.1):
    """
    Overlay labeled boxes on an image with formatted scores and label names.

    Args:
        image -- the image as a numpy array
        boxes -- list of detection boxes
        class_names -- list of classes for each detected object
        scores -- numbers showing the model's confidence in detecting that object
        max_boxes -- maximum detection boxes to overlay on the image (default is 10)
        min_score -- minimum score required to display a bounding box

    Returns:
        image -- the image after detection boxes and classes are overlaid on the original image.
    """
    colors = list(ImageColor.colormap.values())

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSansNarrow-Regular.ttf",
                              25)
    except IOError:
        print("Font not found, using default font.")
        font = ImageFont.load_default()

    for i in range(min(boxes.shape[0], max_boxes)):

        # only display detection boxes that have the minimum score or higher
        if scores[i] >= min_score:
            ymin, xmin, ymax, xmax = tuple(boxes[i])
            display_str = "{}: {}%".format(class_names[i].decode("ascii"),
                                         int(100 * scores[i]))
            color = colors[hash(class_names[i]) % len(colors)]
            image_pil = Image.fromarray(np.uint8(image)).convert("RGB")

            # draw one bounding box and overlay the class labels onto the image
            draw_bounding_box_on_image(image_pil,
                                       ymin,
                                       xmin,
                                       ymax,
                                       xmax,
                                       color,
                                       font,
                                       display_str_list=[display_str])
            np.copyto(image, np.array(image_pil))

    return image

### run_detector

This function will take in the object detection model `detector` and the path to a sample image, then use this model to detect objects.
- This time, run_dtector also calls `draw_boxes` to draw the predicted bounding boxes.

In [ ]:
def load_img(path):
    '''
    Loads a JPEG image and converts it to a tensor.

    Args:
        path (string) -- path to a locally saved JPEG image

    Returns:
        (tensor) -- an image tensor
    '''

    # read the file
    img = tf.io.read_file(path)

    # convert to a tensor
    img = tf.image.decode_jpeg(img, channels=3)

    return img


def run_detector(detector, path):
    '''
    Runs inference on a local file using an object detection model.

    Args:
        detector (model) -- an object detection model loaded from TF Hub
        path (string) -- path to an image saved locally
    '''

    # load an image tensor from a local file path
    img = load_img(path)

    # add a batch dimension in front of the tensor
    converted_img  = tf.image.convert_image_dtype(img, tf.float32)[tf.newaxis, ...]

    # run inference using the model
    start_time = time.time()
    result = detector(converted_img)
    end_time = time.time()

    # save the results in a dictionary
    result = {key:value.numpy() for key,value in result.items()}

    # print results
    print("Found %d objects." % len(result["detection_scores"]))
    print("Inference time: ", end_time-start_time)

    # draw predicted boxes over the image
    image_with_boxes = draw_boxes(
      img.numpy(), result["detection_boxes"],
      result["detection_class_entities"], result["detection_scores"])

    # display the image
    display_image(image_with_boxes)

### Run the detector on your selected image!

In [ ]:
run_detector(detector, downloaded_image_path)

### Run the detector on more images
Perform inference on some additional images of your choice and check how long inference takes.

In [ ]:
image_urls = [
  # Source: https://commons.wikimedia.org/wiki/File:The_Coleoptera_of_the_British_islands_(Plate_125)_(8592917784).jpg
  "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQpfeiclknAq9FEwcLYkHn3noEf1n45R3M9IJaXoMR5sA&s=10",
  # By Américo Toledano, Source: https://commons.wikimedia.org/wiki/File:Biblioteca_Maim%C3%B3nides,_Campus_Universitario_de_Rabanales_007.jpg
  "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQG4VOWcm2wLqC0zQTxA8xNYk86xcijMMM5AhwakQjArw&s=10",
  # Source: https://commons.wikimedia.org/wiki/File:The_smaller_British_birds_(8053836633).jpg
  "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQG4VOWcm2wLqC0zQTxA8xNYk86xcijMMM5AhwakQjArw&s=10",
  ]

def detect_img(image_url):
    start_time = time.time()
    image_path = download_and_resize_image(image_url, 640, 480)
    run_detector(detector, image_path)
    end_time = time.time()
    print("Inference time:",end_time-start_time)

In [ ]:
detect_img(image_urls[0])

In [ ]:
detect_img(image_urls[1])

In [ ]:
detect_img(image_urls[2])